### A More Visible Agent Loop

In [3]:
# Importing necessary libraries
from rich.console import Console
from dotenv import load_dotenv
from openai import AzureOpenAI
import json
from IPython.display import Markdown, display
load_dotenv(override=True)

True

In [20]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")


AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable


In [28]:
# prepare the client for OpenAI
client_OpenAI = AzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT
)

In [4]:
## let's create a function for printing things in a formatted way of console
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [7]:
# let's define some lists for tracking
checklist = []
completed = []

In [8]:
# create a function for getting checklist report
def get_checklist_report() -> str:
    result = ''
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result


In [9]:
get_checklist_report()

''

In [10]:
# function to create checklist
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()


In [11]:
# function to mark checklist complete
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index"
    Console().print(completion_notes)
    return get_checklist_report()


In [12]:
# clear list and let's create some checklist
create_checklist(['Buy groceries','Finish week 1','Eat banana'])


Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [13]:
mark_complete(1, "Bought")

Bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [ ]:
# Let's create these functions as in tool format for AI Model
# define a json info for AI model on how to use tool (OpenAI)
create_checklist_json_openai = {
    "name":"create_checklist",
    "description":"Add new checklist from the list of descriptions and return the full list",
    "parameters":{
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {
                    "type": "string"
                },
                "title": "Description of checklist items"
            }
        },
        "required":["descriptions"],
        "additionalProperties":False
    }
}
mark_complete_json_openai = {
    "name":"mark_complete",
    "description":"Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters":{
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The 1-based index of the checklist item to makr as complete",
                "title": "Index"
            },
            "completion_notes": {
                "type": "string",
                "description": "Note about how you completed the checklist item in rich console markup",
                "title": "Completion Notes"
            }
        },   
        "required":["index","completion_notes"],
        "type": "object",
        "additionalProperties":False
    }
}
# combine tools
tools_openai = [
    {
        "type": "function",
        "function": create_checklist_json_openai
    },
    {
        "type": "function",
        "function": mark_complete_json_openai
    }
]
print(tools_openai)

[{'type': 'function', 'function': {'name': 'create_checklist', 'description': 'Add new checklist from the list of descriptions and return the full list', 'parameters': {'type': 'object', 'properties': {'descriptions': {'type': 'array', 'items': {'type': 'string'}, 'title': 'Description of checklist items'}}, 'required': ['descriptions'], 'additionalProperties': False}}}, {'type': 'function', 'function': {'name': 'mark_complete', 'description': 'Mark complete the checklist item at the given position (starting from 1) and return the full list', 'parameters': {'type': 'object', 'properties': {'index': {'type': 'integer', 'description': 'The 1-based index of the checklist item to makr as complete', 'title': 'Index'}, 'completion_notes': {'type': 'string', 'description': 'Note about how you complted the checklist item in rich console markup', 'title': 'Completion Notes'}}, 'required': ['index', 'completion_notes'], 'additionalProperties': False}}}]


In [16]:
## Let's make a function to handle tool calls
## It will take a list of tool calls, and run them. This is the replacement for that code which we put after knowing AI need tools
## tool handling function for openAI
def handle_tool_calls_openai(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: '{tool_name}' with arguments '{arguments}'", flush=True)
        # Using globals to call the tool function
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id
            }
        )
    return results

## Tool handling function for Anthropic
def handle_tool_calls_anthropic(response_content):
    results = [] # define an empty list to store the results of the tool
    for content in response_content:
        if content.type == "tool_use":
            tool_name, tool_inputs, tool_id = content.name, content.input, content.id
            print(f"Tool called: '{tool_name}' with arguments '{tool_inputs}'", flush=True)
            # Using globals to call the tool function
            tool = globals().get(tool_name)
            result = tool(**tool_inputs) if tool else {}
            # storing tool results                
            results.append({
                "type":"tool_result",
                "content": str(result),
                "tool_use_id": tool_id
            })
    return results

In [18]:
# define a system prompt
system_prompt = """
You are  given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, ste the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask user questions or clarification; respond only with the answer after using your tools.
"""
user_prompt = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph towards Boston.
When do they meet.
"""

In [33]:
# Let's update this method for calling for AI model to chat with tool
def ask_AI_model(user_prompt, history, system_prompt: str = system_prompt, client: str = "openai", OpenAI_deployment: str = AZURE_OPENAI_DEPLOYMENT_GPT_55, 
    tools_openai:list = tools_openai) -> str:
    if client == "OpenAI" or client == "openAI" or client == "openai":
        # let's set the AI messages
        messages = [
            {
                "role": "system",
                "content": system_prompt
            }] + history + [
            {
                "role": "user",
                "content": user_prompt
            }
        ]
        # seeking response from AI model
        response = client_OpenAI.chat.completions.create(
        model=OpenAI_deployment,
        messages=messages,
        tools=tools_openai
        )
        # set the condition for tools calling
        while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message) # appnending the message to the chat history
            tool_calls = message.tool_calls
            results = handle_tool_calls_openai(tool_calls)
            messages.extend(results)
            # calling AI model second time
            response = client_OpenAI.chat.completions.create(
            model=OpenAI_deployment,
            messages=messages,
            tools=tools_openai
            )
        show(response.choices[0].message.content)

    else:
        return "PLEASE GIVE A CHECK ON CLIENT NAME BEFORE CALLING FUNCTION"

In [34]:
checklist, completed = [],[]
ask_AI_model(
    user_prompt, 
    []
)

Tool called: 'create_checklist' with arguments '{'descriptions': ['Identify the needed route distance between Boston and New York; since it is not provided, use a reasonable estimate.', 'Compute how far the Boston train travels before the New York train departs.', 'Compute the remaining distance between trains at 3:00 pm.', 'Use the combined speed after 3:00 pm to find time until they meet.', 'Convert the elapsed time into a clock time and present the answer.']}'


Checklist #1: Identify the needed route distance between Boston and New York; since it is not provided, use a 
reasonable estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between trains at 3:00 pm.
Checklist #4: Use the combined speed after 3:00 pm to find time until they meet.
Checklist #5: Convert the elapsed time into a clock time and present the answer.

Tool called: 'mark_complete' with arguments '{'index': 1, 'completion_notes': '[green]Distance was not provided, so I used a reasonable train-route estimate of about [bold]230 miles[/bold] between Boston and New York.[/green]'}'


Distance was not provided, so I used a reasonable train-route estimate of about 230 miles between Boston and New 
York.

Checklist #1: Identify the needed route distance between Boston and New York; since it is not provided, use a 
reasonable estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between trains at 3:00 pm.
Checklist #4: Use the combined speed after 3:00 pm to find time until they meet.
Checklist #5: Convert the elapsed time into a clock time and present the answer.

...

Checklist #1: Identify the needed route distance between Boston and New York; since it is not provided, use a 
reasonable estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between trains at 3:00 pm.
Checklist #4: Use the combined speed after 3:00 pm to find time until they meet.
Checklist #5: Convert the elapsed time into a clock time and present the answer.

Tool called: 'mark_complete' with arguments '{'index': 5, 'completion_notes': '[green]Adding about 1 hour 13 minutes to 3:00 pm gives a meeting time of approximately [bold]4:13 pm[/bold].[/green]'}'


Adding about 1 hour 13 minutes to 3:00 pm gives a meeting time of approximately 4:13 pm.

Checklist #1: Identify the needed route distance between Boston and New York; since it is not provided, use a 
reasonable estimate.
Checklist #2: Compute how far the Boston train travels before the New York train departs.
Checklist #3: Compute the remaining distance between trains at 3:00 pm.
Checklist #4: Use the combined speed after 3:00 pm to find time until they meet.
Checklist #5: Convert the elapsed time into a clock time and present the answer.

Assuming the Boston–New York train-route distance is about 230 miles:

• Boston train leaves at 2:00 pm at 60 mph.  
• By 3:00 pm, it has traveled:

60 × 1 = 60 miles

• Remaining distance between the trains at 3:00 pm:

230 − 60 = 170 miles

• Once the New York train leaves, they move toward each other at:

60 + 80 = 140 mph

• Time to meet after 3:00 pm:

170 ÷ 140 = 1.214 hours ≈ 1 hour 13 minutes

They meet at approximately 4:13 pm.